## Define libraries

In [1]:
import sys
import os


# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [2]:
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# from helper import RAGHelper
from dotenv import load_dotenv
load_dotenv()
import os

## Define Variables

In [3]:
# Constants

DATASET_SOURCE = 'suniltvl/ragbench'
DATA_SPLIT = 'test'
VECTOR_DATABASES = ['chroma'] # ['chroma', 'milvus']
EMBEDDING_MODELS = ["BAAI/LLM-Embedder", "BAAI/bge-large-en-v1.5"]
MAX_CHUNKS = 5000
CHUNKING_SIZES = [256, 512, 1024]
CHUNKING_OVERLAPS = [50, 100, 200]
SEPARATORS = ["\n\n", "\n", " ", ".", ","]
DOMAINS = {
    'cs':
    {
        # "delucionqa":"Jeep manual", 
        # "emanual": "TV manual", 
        # "techqa":"Technotes"
    },
    'gk':
    {
        # "hotpotqa":"wiki 1",
        # "msmacro":"web pages",
        # "hagrid":"wiki 2",
        # "expertqa":"googlesearch"
    }
}

### Check Device

In [4]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.12.0+cpu
None
False


### Vector DB Cache

In [5]:
VECTOR_DB_CACHE = {}

### Templates

In [6]:
from pathlib import Path

# ------------------------------------------------------------------
# Path Templates
# ------------------------------------------------------------------

def get_db_folder(db_type: str) -> Path:
    return Path(f"../database_{db_type}")


def get_collection_name(
    embedding_model: str,
    domain_name_key: str,
) -> str:
    return f"{embedding_model.replace('/', '_')}-{domain_name_key}"


def get_persist_directory(
    db_type: str,
    domain_name_key: str,
    chunk_size: int,
    chunk_overlap: int,
) -> Path:
    return (
        get_db_folder(db_type)
        / f"{domain_name_key}_{chunk_size}_{chunk_overlap}"
    )


### All Variables

In [7]:
# all_db_folders = [get_db_folder(vdb) for vdb in EMBEDDING_MODELS]

# all_collections = []
# for db_folder in EMBEDDING_MODELS:
#     all_collections.extend(list(db_folder.iterdir()))

# all_collections

## Helper Functions

In [8]:
def deduplicate_data(data, doc_type):
    data_dict = {}
    for d in data:
        document = " ".join(d["documents"])
        if document in data_dict:
            data_dict[document]["docid"].append(d["id"])
        else:
            data_dict[document] = {"docid":[d["id"]]}
            
    for k in data_dict:
        data_dict[k]["document_type"] = doc_type
        
    return data_dict

### Data Fetching

In [9]:
def get_docs(db_name: str):
    dataset = load_dataset(DATASET_SOURCE, db_name, split=DATA_SPLIT)
    dedup = deduplicate_data(dataset, db_name)
    docs = [
        Document(
            metadata=metadata, 
            page_content=content
        )
        for content, metadata in dedup.items()
    ]   
    return docs


### Chunking

In [10]:
def chunk_documents(docs: list,chunk_size: int, chunk_overlap: int):    

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=SEPARATORS)
    docs_chunks = text_splitter.split_documents(docs)
    
    return docs_chunks

### Embedding Models

In [11]:
# Use a shared local cache so Hugging Face models are not re-downloaded every run
HF_CACHE_DIR = Path(parent_dir) / ".cache" / "huggingface"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_HOME", str(HF_CACHE_DIR))
os.environ.setdefault("TRANSFORMERS_CACHE", str(HF_CACHE_DIR / "transformers"))
os.environ.setdefault("HF_HUB_CACHE", str(HF_CACHE_DIR / "hub"))

# Reuse embedding instances across the loop to avoid reloading the same model
EMBEDDING_CACHE = {}

def get_embedding_function(model: str):
    if model not in EMBEDDING_CACHE:
        print(f"Loading embedding model: {model}")
        EMBEDDING_CACHE[model] = HuggingFaceEmbeddings(
            model=model,
            # model_kwargs={
            #     "device": "cuda"
            # },
            # encode_kwargs={
            #     "batch_size": 64,
            #     "normalize_embeddings": True,
            # },
            cache_folder=str(HF_CACHE_DIR),
        )
    else:
        print(f"Reusing embedding model from cache: {model}")
    return EMBEDDING_CACHE[model]

### Vectorization

#### Get Vector DB from Cache

In [12]:

def get_vector_db(persist_directory, collection_name, embedding_function):
    key = (persist_directory, collection_name)

    if key not in VECTOR_DB_CACHE:
        VECTOR_DB_CACHE[key] = Chroma(
            persist_directory=persist_directory,
            collection_name=collection_name,
            embedding_function=embedding_function
        )

    return VECTOR_DB_CACHE[key]

In [13]:
def add_document_to_vector_db(vector_db, document):
    vector_db.add_documents([document])
    return vector_db

#### Create Vector Docs

In [14]:
def vectorize_documents(docs_chunks: list, db_name: str, persist_directory: str, collection_name: str, embedding_function):

    db_exists = os.path.exists(persist_directory) and os.listdir(persist_directory)

    if db_exists:
        print(f"Loading existing vector database... {db_name}")
        vector_db = get_vector_db(
            persist_directory,
            collection_name,
            embedding_function,
        )
    else:
        print(f"Creating new vector database... {db_name}")

        first_batch = docs_chunks[:MAX_CHUNKS]

        vector_db = Chroma.from_documents(
            documents=first_batch,
            embedding=embedding_function,
            persist_directory=persist_directory,
            collection_name=collection_name,
        )

        docs_chunks = docs_chunks[MAX_CHUNKS:]

    for i in range(0, len(docs_chunks), MAX_CHUNKS):
        vector_db.add_documents(
            docs_chunks[i:i + MAX_CHUNKS]
        )

    return vector_db

## Looping Domains

In [ ]:
def loop_domains():   
    # Loop through domains and datasets
    for domain_short_name, data_set in DOMAINS.items():

        print(f"Processing domain: {domain_short_name}")        
        # Loop through datasets
        for data_set_path, name in data_set.items():

            print(f"  Processing dataset: {data_set_path} > {name}")
            
            # Read from dataset (HuggingFace)
            docs = get_docs(data_set_path)

            print(f"  Number of documents: {len(docs)} in {data_set_path}")

            # Loop through chunking sizes and overlaps
            for chunk_size, chunk_overlap in zip(CHUNKING_SIZES, CHUNKING_OVERLAPS):

                print(f"   Processing chunk size: {chunk_size}, overlap: {chunk_overlap}")

                # Chunk documents
                chunks = chunk_documents(docs, chunk_size, chunk_overlap)

                print(f"   Number of chunks: {len(chunks)} in {data_set_path} for chunk size: {chunk_size}, overlap: {chunk_overlap}")

                # Loop Embedder function
                for embedder in EMBEDDING_MODELS:
                    print(f"      Processing embedder: {embedder}")
                    # Generate embeddings
                    embedding_function = get_embedding_function(embedder)

                    # Loop Vector Database
                    for vector_db_type in VECTOR_DATABASES:
                        print(f"         Processing vector database: {vector_db_type} for embedder: {embedder}, chunk size: {chunk_size}, overlap: {chunk_overlap}, data set: {data_set_path}, domain: {domain_short_name}")

                        # Get DB Name
                        db_name = get_db_folder(vector_db_type)
                        
                        # Get Collection Name
                        collection_name = get_collection_name(embedder, domain_short_name)

                        # Get persist directory
                        persist_directory = get_persist_directory(vector_db_type, domain_short_name, chunk_size, chunk_overlap)                    

                        print(f"         Persist directory: {persist_directory}, Collection name: {collection_name}, db_name: {db_name}")
                        vector_db = vectorize_documents(chunks, db_name, persist_directory, collection_name, embedding_function)

                    


In [ ]:
loop_domains()

## Test Methods

In [25]:
# from utils.helper import *

persist_directory = get_persist_directory(VECTOR_DATABASES[0],"cs", 256, 50)
collection_name = get_collection_name(EMBEDDING_MODELS[0], "cs")
embedding_function = get_embedding_function(EMBEDDING_MODELS[0])

print(persist_directory)
print(collection_name)
print(embedding_function)
vdb = get_vector_db(persist_directory, collection_name, embedding_function)
print(vdb)


Reusing embedding model from cache: BAAI/LLM-Embedder
..\database_chroma\cs_256_50
BAAI_LLM-Embedder-cs
model_name='BAAI/LLM-Embedder' cache_folder='f:\\Github\\Capstone\\nextgen-rag-system\\.cache\\huggingface' model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [29]:
document_types = set()

offset = 0
batch_size = 1000

while True:
    results = vdb._collection.get(
        # where={"document_type": "emanual"},
        include=["metadatas"],
        limit=batch_size,
        offset=offset,
    )

    if not results["metadatas"]:
        break

    for metadata in results["metadatas"]:
        if metadata and "docid" in metadata:
            document_types.add(metadata["docid"][0])

    offset += batch_size

print(results)

{'ids': [], 'embeddings': None, 'documents': None, 'uris': None, 'included': ['metadatas'], 'data': None, 'metadatas': []}


In [27]:
results = vdb._collection.get(
        # where={"document_type": "techqa"},
        include=["metadatas"],
        limit=1,
        offset=0,
    )
print(results)

{'ids': ['277d2a4d-3ac2-433f-8bb7-839e9413fe3c'], 'embeddings': None, 'documents': None, 'uris': None, 'included': ['metadatas'], 'data': None, 'metadatas': [{'document_type': 'delucionqa', 'docid': ['114', '114']}]}
